# 03 — Causal feature engineering

**Prediction instant:** end of each one-minute interval. The current-minute mean and first difference are available at that instant. Lag-1, rolling mean/std/min/max are calculated strictly from earlier minutes. These are anomaly-detection features, not future-failure labels. Window = 12 past minutes; compute separately within each split to avoid boundary mixing.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import RAW, PROCESSED, MODELS, ANALOG, DIGITAL, FAILURES

from src.preprocessing.preprocess import temporal_split
from src.features.build_features import make_features
df=pd.read_pickle(PROCESSED/'cleaned.pkl');train,val,test=temporal_split(df)
parts={k:make_features(v) for k,v in [('train',train),('val',val),('test',test)]}
for k,v in parts.items():print(k,v.shape)
display(parts['train'].head())

## Feature interpretation

`*_mean`: current-minute sensor level; `*_lag1`: previous minute level; `*_rolling_mean`: historical operating level; `*_rolling_std`: short-term volatility; `*_rolling_min/max`: recent range; `*_delta`: minute-to-minute change. Digital channel minute means indicate fraction of active samples. Neither raw timestamps nor original row IDs enter the model. Sparse/gap minutes can yield missing features, handled by training-fitted imputation.

In [ ]:
for name,frame in parts.items():frame.to_pickle(PROCESSED/f'{name}_features.pkl')
print('Feature columns:',[c for c in parts['train'] if c!='timestamp'])